# D76 基线4: IQL(离线, 专家分片, D64.2 修复版)——`base_iql_s0`

先重生成当前 env 口径的离线数据(死锁手术后, 旧数据作废), 再训。注: 离线线用 fv1 特征(数据生成口径), 论文注明。定位=带优势倾斜的 BC 对照。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 重生成离线数据(70核, ~几分钟)

In [ ]:
!{PREFIX} scripts/33_build_offline_dataset.py train 24 70 5000 0 6 12 v2   # 新 env 口径重生成 -> offline_shards_train_c12v2


## ② 训练 IQL

In [ ]:
!{PREFIX} scripts/34_train_iql.py \
    --dataset artifacts/dataset/offline_shards_train_c12v2 \
    --shard-glob 'signaller_*.pkl' --control-classes 12 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --iters 1500 --batch 128 --eval-every 150 --gamma 0.99 --tau 0.7 --beta 0.5 \
    --run-tag base_iql_s0 --resume
